# CellPose-SAM — Custom Model Training

Fine-tunes a CellPose-SAM model on your own annotated data.

## Workflow

1. **Export annotations from QuPath** using the scripts in `export_annotations/cellpose_training/`.  
   Each exported image produces a pair of files: `<name>_img.tif` and `<name>_mask.tif`.

2. **Organise your data** — place the exported pairs in the `data/` folder next to this notebook, either flat or in subdirectories (one per dataset):
   ```
   data/
   ├── frame001_img.tif        ← flat: all images in data/
   └── frame001_mask.tif
   ```
   ```
   data/
   ├── experiment_A/           ← multi-dataset: one subfolder per dataset
   │   ├── frame001_img.tif
   │   └── frame001_mask.tif
   └── experiment_B/
       ├── frame001_img.tif
       └── frame001_mask.tif
   ```

3. **Create the train/test split** — run from the terminal (with the `cpsam` environment active):
   ```
   python split_data.py data/
   ```
   This creates `data/splits/train/` and `data/splits/test/` (90 / 10 % by default).

4. **Edit the configuration cell** (Cell 3) — set your model name and adjust hyperparameters if needed.

5. **Run all cells** — the trained model is saved to `models/<model_name>`.

---

> **3D data:** multi-page TIFF stacks are supported without any extra configuration — cellpose auto-detects dimensionality from the arrays. When running inference on 3D stacks, use `model.eval(..., do_3D=True)`.

In [ ]:
from cellpose import io, models, train
import matplotlib.pyplot as plt

In [ ]:
# ── Edit this cell ──────────────────────────────────────────────────────────
model_name    = "my_cpsam_model"   # name for the saved model
train_dir     = "data/splits/train"
test_dir      = "data/splits/test"
n_epochs      = 400
learning_rate = 1e-5
weight_decay  = 0.1
use_gpu       = True               # set to False to use CPU
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
io.logger_setup()

output = io.load_train_test_data(
    train_dir, test_dir,
    image_filter="_img", mask_filter="_mask",
)
images, labels, _, test_images, test_labels, _ = output

model = models.CellposeModel(gpu=use_gpu)

model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=images,
    train_labels=labels,
    test_data=test_images,
    test_labels=test_labels,
    weight_decay=weight_decay,
    learning_rate=learning_rate,
    n_epochs=n_epochs,
    model_name=model_name,
)

print(f"Model saved to: {model_path}")

In [ ]:
from pathlib import Path

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_losses, label="train")
    ax.plot(test_losses, label="test")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    fig.tight_layout()
    png_path = Path("models") / f"{model_name}_loss.png"
    png_path.parent.mkdir(exist_ok=True)
    fig.savefig(png_path, dpi=150)
    plt.show()
    print(f"Loss plot saved to: {png_path}")
except ImportError:
    print("matplotlib not available — install it to plot losses.")
    print(f"train_losses: {train_losses}")
    print(f"test_losses:  {test_losses}")